In [1]:
#BGE sample

from FlagEmbedding import FlagModel

model = FlagModel(
    'BAAI/bge-large-zh-v1.5',
    query_instruction_for_retrieval="为这个句子生成表示以用于检索：",
    use_fp16=True
)

sentences_1 = ["我爱自然语言处理", "我喜欢机器学习"]
sentences_2 = ["我热爱BGE模型", "我关注文本检索"]

embeddings_1 = model.encode(sentences_1)
embeddings_2 = model.encode(sentences_2)

import numpy as np
similarity = np.matmul(embeddings_1, embeddings_2.T)
print(similarity)


/usr/local/miniconda3/envs/AIaplication_pro/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


[[0.4207 0.4873]
 [0.519  0.562 ]]


In [4]:
import os, json, random
from glob import glob

random.seed(42)

# === 输入输出路径 ===
DATA_DIR = "CSTS"
OUT_DIR = "CSTS_BGE"
os.makedirs(OUT_DIR, exist_ok=True)

def read_txt_file(path):
    """读取 CSTS 的 txt 文件，返回 [(s1, s2, label), ...]"""
    pairs = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) < 3:
                continue
            s1, s2, label = parts[0].strip(), parts[1].strip(), parts[2].strip()
            if not s1 or not s2:
                continue
            # 部分文件可能label是'1.0'或'True'等，统一为0/1
            label = label.strip().lower()
            if label in ['1', '1.0', 'true', 't']:
                lab = 1
            elif label in ['0', '0.0', 'false', 'f']:
                lab = 0
            else:
                try:
                    lab = int(float(label) > 0.5)
                except:
                    continue
            pairs.append((s1, s2, lab))
    return pairs

def build_jsonl(split='train'):
    """整合 AFQMC, LCQMC, OPPO-xiaobu 三个子集生成统一 JSONL"""
    all_samples = []
    for sub in ['AFQMC', 'LCQMC', 'OPPO-xiaobu']:
        path = os.path.join(DATA_DIR, sub, f'{split}.txt')
        if not os.path.exists(path):
            print(f"⚠️ {path} 不存在，跳过")
            continue
        data = read_txt_file(path)
        all_samples.extend(data)
        print(f"✅ 读取 {sub}/{split}.txt: {len(data)} 条")

    print(f"➡️ 合并后 {split} 样本总数：{len(all_samples)}")

    # === 构建对比学习格式 ===
    output_path = os.path.join(OUT_DIR, f'{split}.jsonl')
    with open(output_path, 'w', encoding='utf-8') as f:
        for i, (q, pos, lab) in enumerate(all_samples):
            if lab == 1:
                # 随机抽取两个负样本
                negs = []
                for _ in range(2):
                    n_q, n_pos, n_lab = random.choice(all_samples)
                    if n_lab == 0:
                        negs.append(n_pos)
                f.write(json.dumps({
                    "query": q,
                    "pos": [pos],
                    "neg": negs
                }, ensure_ascii=False) + '\n')

    print(f"✅ 已保存 {output_path}")
    print("—"*30)

# === 执行 ===
build_jsonl('train')
build_jsonl('dev')


✅ 读取 AFQMC/train.txt: 34334 条
✅ 读取 LCQMC/train.txt: 238766 条
✅ 读取 OPPO-xiaobu/train.txt: 167168 条
➡️ 合并后 train 样本总数：440268
✅ 已保存 CSTS_BGE/train.jsonl
——————————————————————————————
✅ 读取 AFQMC/dev.txt: 4316 条
✅ 读取 LCQMC/dev.txt: 8802 条
✅ 读取 OPPO-xiaobu/dev.txt: 10000 条
➡️ 合并后 dev 样本总数：23118
✅ 已保存 CSTS_BGE/dev.jsonl
——————————————————————————————


In [5]:
from FlagEmbedding import FlagModel, PairDataset
from torch.utils.data import DataLoader
from transformers import AdamW

model = FlagModel('BAAI/bge-large-zh-v1.5', use_fp16=True)
train_dataset = PairDataset("CSTS_BGE/train.jsonl")
dev_dataset   = PairDataset("CSTS_BGE/dev.jsonl")

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=4)
optimizer = AdamW(model.parameters(), lr=2e-5)

for epoch in range(2):
    for step, batch in enumerate(train_loader):
        loss = model.compute_loss(batch)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        if step % 200 == 0:
            print(f"Epoch {epoch} Step {step} Loss={loss.item():.4f}")

model.save("bge-large-zh-v1.5-csts-finetuned")


/usr/local/miniconda3/envs/AIaplication_pro/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: cannot import name 'PairDataset' from 'FlagEmbedding' (/root/Code/AIaplication_pro/FlagEmbedding_src/FlagEmbedding/__init__.py)